# 01 · Data quality and preparation

All data is **synthetic** (fictional person). This notebook re-runs the cleaning pipeline (`analysis/scripts/clean_data.py`) and inspects its report: types, missing values, duplicates, label/unit standardisation, invalid values, broken relationships and join safety.

In [1]:
import subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
print(subprocess.run([sys.executable, str(ROOT/'analysis/scripts/clean_data.py')], capture_output=True, text=True).stdout)

Clean tables written to C:\Users\Pavan kumar\Food Health\data\processed
Rejected rows: 7  |  report: analysis/reports/data_quality_report.md



In [2]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))
import pandas as pd, matplotlib.pyplot as plt
from foodspend_core import analytics as A
from foodspend_core.recommender import recommend, simulate, budget_comparison
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
FIG = ROOT / 'analysis' / 'reports' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e1e0d9',
                     'axes.edgecolor': '#c3c2b7', 'font.size': 10, 'figure.dpi': 110})
ds = A.load_processed(ROOT / 'data' / 'processed')
print('SYNTHETIC DATA - generated by analysis/scripts/generate_synthetic_data.py')

SYNTHETIC DATA - generated by analysis/scripts/generate_synthetic_data.py


In [3]:
report = json.loads((ROOT/'analysis/reports/data_quality_report.json').read_text())
pd.DataFrame([{'table': k, 'raw_rows': v['raw_rows'], 'clean_rows': v['clean_rows'], 'missing': v['missing_values']} for k, v in report['tables'].items()])

,table,raw_rows,clean_rows,missing
0,users,1,1,{}
1,vendors,9,9,{}
2,foods,40,40,{}
3,food_nutrition,40,40,{'fiber_g': 2}
4,meal_options,80,80,{}
5,orders,1225,1223,{'meal_type': 73}
6,order_items,1762,1755,"{'food_id': 4, 'portion_consumed': 330}"
7,food_feedback,208,206,"{'order_id': 3, 'rating': 1}"
8,health_measurements,81,79,{'notes': 12}
9,nutrition_goals,3,3,{}


## Checks and actions taken
Rows are rejected (never silently fixed by guesswork) when quantities are non-positive, foods/orders are unknown, ratings fall outside 1-5, or measurements are implausible.

In [4]:
pd.DataFrame(report['checks']).fillna('')

C:\Users\Pavan kumar\AppData\Local\Temp\ipykernel_8316\1012144501.py:1: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  pd.DataFrame(report['checks']).fillna('')


,table,check,rows_removed,mismatches,foods,rows,rule,order_ids,difference_inr,variants,resolved,unresolved,rows_rejected,orders_checked,orders_without_items,feedback_ids
0,orders,exact duplicate rows removed,2.0,,,,,,,,,,,,,
1,order_items,exact duplicate rows removed,3.0,,,,,,,,,,,,,
2,health_measurements,exact duplicate rows removed,1.0,,,,,,,,,,,,,
3,food_nutrition,nutrition basis unit matches food serving unit,,0.0,,,,,,,,,,,,
4,food_nutrition,foods with unknown nutrient values (kept as NU...,,,"[F013: fiber_g, F016: fiber_g]",,,,,,,,,,,
5,orders,meal_type inferred from order time (flagged me...,,,,2.0,"breakfast <11:00, lunch 11-16, snack 16-19, di...",,,,,,,,,
6,orders,final_total recomputed from components (subtot...,,,,2.0,,"[O00017, O00676]",100.0,,,,,,,
7,order_items,serving units standardised,,,,30.0,,,,"[G, ML, gm, grams]",,,,,,
8,order_items,blank food_id resolved by exact normalised-nam...,,,,,,,,,3.0,1.0,,,,
9,order_items,raw food names standardised to catalogue name,,,,43.0,,,,,,,,,,


In [5]:
rej = pd.read_csv(ROOT/'data/processed/rejected_records.csv')
rej[['table','record_id','rule']]

,table,record_id,rule
0,order_items,I001735,unknown food (orphan food_id)
1,order_items,I001759,unknown food (orphan food_id)
2,order_items,I001757,quantity must be > 0
3,order_items,I001758,quantity must be > 0
4,food_feedback,FB0006,rating missing or outside 1-5
5,food_feedback,FB0018,rating missing or outside 1-5
6,health_measurements,H0021,physiologically implausible value or unknown i...


## Data types after cleaning

In [6]:
ds.orders.dtypes.to_frame('dtype').T

,order_id,user_id,order_datetime,meal_type,record_type,vendor_id,items_subtotal_inr,order_discount_inr,tax_inr,delivery_fee_inr,other_charges_inr,final_total_inr,order_date,meal_type_inferred,total_recomputed,counts_as_spending
dtype,str,str,datetime64[us],str,str,str,float64,float64,float64,float64,float64,float64,datetime64[us],bool,bool,bool


## Relationship and join checks
Order charges live on `orders`; summing them after joining to `order_items` would duplicate them for multi-item orders.

In [7]:
so = A.spending_orders(ds)
naive = so.merge(ds.items, on='order_id')['final_total_inr'].sum()
print(f'Correct order-level total: Rs {so.final_total_inr.sum():,.2f}')
print(f'Naive item-join total:     Rs {naive:,.2f}  (overstated)')
recon = ds.items.groupby('order_id').line_total_inr.sum().round(2)
print('Orders where item lines != subtotal:', int(((recon - ds.orders.set_index('order_id').items_subtotal_inr.reindex(recon.index)).abs() > 0.01).sum()))
print('Orphan items:', int((~ds.items.order_id.isin(ds.orders.order_id)).sum()), '| items with unknown food:', int((~ds.items.food_id.isin(ds.foods.food_id)).sum()))

Correct order-level total: Rs 148,988.10
Naive item-join total:     Rs 289,342.67  (overstated)
Orders where item lines != subtotal: 0
Orphan items: 0 | items with unknown food: 0


## Coverage

In [8]:
report['coverage']

{'first_date': '2025-10-27',
 'last_date': '2026-10-26',
 'calendar_days': 365,
 'days_with_records': 346,
 'gaps_of_3_plus_days': [{'from': '2026-03-14',
   'to': '2026-03-29',
   'days': 16}],
 'orders_by_record_type': {'home_meal': 575,
  'delivery_order': 431,
  'cafe_purchase': 146,
  'grocery_purchase': 71},
 'items_by_consumption_status': {'consumed': 1321,
  'unknown': 326,
  'partially_consumed': 86,
  'not_consumed': 22}}

In [9]:
pd.DataFrame(report['health_frequency']).T

,count,first,last,median_days_between
diastolic_bp,11,2025-11-03,2026-10-02,31.5
fasting_glucose,4,2025-11-10,2026-08-24,95.0
systolic_bp,11,2025-11-03,2026-10-02,31.5
waist_circumference,12,2025-11-03,2026-10-02,30.0
weight,41,2025-11-02,2026-10-25,7.0
